# Pipeline Smoke Test

This notebook runs a deliberately tiny end-to-end pass through the forecasting and RL pipeline. 
It is meant to catch integration errors before starting the expensive full experiment.


In [1]:
%load_ext autoreload
%autoreload 2


## Imports And Paths


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "fyp_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from fyp_pipeline.core_pipeline import CONFIG, configure_vast_ai, prepare_data, print_task_summary
from fyp_pipeline.experiment_runner import (
    run_experiment,
    build_cl_summary,
    print_and_save_comparison_tables,
)
from fyp_pipeline.trainers import LOGGER, compute_mase


Project root: c:\Users\Syakir\Downloads\Projects\fyp


c:\Users\Syakir\Downloads\Projects\fyp\.venv\Lib\site-packages\pytorch_forecasting\models\base\_base_model.py:30: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


## Runtime Setup


In [3]:
DATA_DIR = str(PROJECT_ROOT / "data" / "processed")
OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / "smoke_test")

configure_vast_ai(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    require_gpu=False,  # set True on Vast.ai if you want to require CUDA
)


Runtime diagnostics

Python         : 3.11.9

PyTorch        : 2.3.1+cpu

CUDA available : False

CPU cores      : 12

Vast.ai        : NO

Active device  : CPU

Precision      : 32

{'paths': {'demand_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\demand_forecasting.csv',
  'rl_csv': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\data\\processed\\rl_environment.csv',
  'checkpoints': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\smoke_test\\checkpoints',
  'results': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\smoke_test\\results',
  'logs': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\smoke_test\\logs',
  'plots': 'C:\\Users\\Syakir\\Downloads\\Projects\\fyp\\outputs\\smoke_test\\plots'},
 'tasks': [{'task_id': 1,
   'name': 'Baseline_2023_H1',
   'start': '2023-01-01',
   'end': '2023-05-31',
   'regime': 'baseline'},
  {'task_id': 2,
   'name': 'MegaSale_2023',
   'start': '2023-06-01',
   'end': '2023-12-31',
   'regime': 'mega_sale'},
  {'task_id': 3,
   'name': 'Baseline_2024_H1',
   'start': '2024-01-01',
   'end': '2024-05-31',
   'regime': 'baseline'},
  {'task_id': 4,
   'name': 'MegaSale_2024',
   'star

## Tiny Smoke-Test Configuration


In [4]:
# Keep this tiny. The goal is correctness, not final metrics.
CONFIG["tasks"] = CONFIG["tasks"][:2]
CONFIG["model_types"] = ["forecasting", "rl"]
CONFIG["cl_methods"] = {
    "forecasting": ["naive", "ewc", "replay", "sdft"],
    "rl": ["naive", "ewc", "recall", "sdft"],
}

CONFIG["forecasting"].update({
    "encoder_length": 28,
    "prediction_length": 7,
    "hidden_size": 16,
    "attention_head_size": 1,
    "hidden_continuous_size": 8,
    "batch_size": 64,
    "max_epochs": 1,
    "early_stop_patience": 1,
})

CONFIG["rl"].update({
    "total_timesteps_per_task": 256,
    "eval_episodes": 1,
    "n_steps": 128,
    "batch_size": 64,
    "n_epochs": 1,
    "net_arch": [32, 32],
})

CONFIG["cl"].update({
    "ewc_fisher_samples": 2,
    "replay_buffer_size": 128,
    "recall_buffer_capacity": 256,
    "recall_mix_n_steps": 32,
})

CONFIG["hardware"].update({
    "compile": False,
    "num_workers": 0,
    "persistent_workers": False,
})

print("Smoke-test config ready")
print("Tasks:", [t["name"] for t in CONFIG["tasks"]])
print("Forecast methods:", CONFIG["cl_methods"]["forecasting"])
print("RL methods:", CONFIG["cl_methods"]["rl"])


Smoke-test config ready
Tasks: ['Baseline_2023_H1', 'MegaSale_2023']
Forecast methods: ['naive', 'ewc', 'replay', 'sdft']
RL methods: ['naive', 'ewc', 'recall', 'sdft']


## Metric Sanity Check


In [5]:
mase_value = compute_mase([2, 3, 4], [2, 2, 5], list(range(20)), seasonality=7)
assert mase_value == mase_value and mase_value > 0, mase_value
print("MASE sanity check:", mase_value)


MASE sanity check: 0.09523809523809523


## Load Data


In [6]:
tft_tasks, rl_tasks, tft_df, rl_df = prepare_data()
print_task_summary(tft_tasks, rl_tasks)

assert len(tft_tasks) == len(CONFIG["tasks"])
assert len(rl_tasks) == len(CONFIG["tasks"])
assert all(len(df) > 0 for df in tft_tasks), "At least one TFT task is empty"
assert all(len(df) > 0 for df in rl_tasks), "At least one RL task is empty"
print("Data checks passed")


Loading datasets...

Demand CSV  : 9,864 rows × 66 cols

RL CSV      : 9,864 rows × 86 cols

✓ Data loaded and cleaned

┏━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Task ┃ Name             ┃ Period                   ┃ TFT rows ┃ RL rows ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ 1    │ Baseline_2023_H1 │ 2023-01-01 -> 2023-05-31 │ 1,359    │ 1,359   │
│ 2    │ MegaSale_2023    │ 2023-06-01 -> 2023-12-31 │ 1,926    │ 1,926   │
└──────┴──────────────────┴──────────────────────────┴──────────┴─────────┘

Data checks passed


## Run Smoke Test


In [7]:
run_experiment(tft_tasks, rl_tasks)
print("Smoke-test training loop completed")


==============================================================

  CONTINUAL LEARNING EXPERIMENT START

==============================================================

═══ MODEL TYPE: FORECASTING ═══

  ── CL Method: naive ──

...

Eval task 1: avg_episode_reward=5994.6390  cumulative_profit=43381647.5254  pricing_regret=31.8141

Eval task 2: avg_episode_reward=7701.6213  cumulative_profit=45146278.6459  pricing_regret=21.8422

Results saved → C:\Users\Syakir\Downloads\Projects\fyp\outputs\smoke_test\results\all_metrics.csv

═══ EXPERIMENT COMPLETE (2.2 min) ═══

Smoke-test training loop completed


## Validate Results


In [8]:
results_df = LOGGER.to_dataframe()
display(results_df.tail(20))

assert not results_df.empty, "No metrics were logged"
expected_model_types = set(CONFIG["model_types"])
assert expected_model_types.issubset(set(results_df["model_type"])), results_df["model_type"].unique()

forecast_df = results_df[results_df["model_type"] == "forecasting"]
rl_df_results = results_df[results_df["model_type"] == "rl"]
assert not forecast_df.empty, "No forecasting metrics logged"
assert not rl_df_results.empty, "No RL metrics logged"

# MASE can be NaN if a deliberately tiny smoke split has insufficient scale,
# but sMAPE/RMSE and RL metrics should exist.
assert {"smape", "rmse"}.issubset(set(forecast_df["metric_name"])), forecast_df["metric_name"].unique()
assert "cumulative_profit" in set(rl_df_results["metric_name"]), rl_df_results["metric_name"].unique()

print("Logged metrics:")
print(results_df.groupby(["model_type", "cl_method", "metric_name"]).size())
print("Smoke test passed")


,model_type,cl_method,train_task_id,eval_task_id,metric_name,metric_value,timestamp
52,rl,ewc,2,2,cumulative_profit,4.448476e+07,2026-05-06T22:31:52
53,rl,ewc,2,2,pricing_regret,2.184758e+01,2026-05-06T22:31:52
54,rl,recall,1,1,avg_episode_reward,6.017215e+03,2026-05-06T22:31:53
55,rl,recall,1,1,cumulative_profit,4.320147e+07,2026-05-06T22:31:53
56,rl,recall,1,1,pricing_regret,3.179759e+01,2026-05-06T22:31:53
57,rl,recall,2,1,avg_episode_reward,5.864445e+03,2026-05-06T22:31:54
58,rl,recall,2,1,cumulative_profit,4.224136e+07,2026-05-06T22:31:54
59,rl,recall,2,1,pricing_regret,3.190956e+01,2026-05-06T22:31:54
60,rl,recall,2,2,avg_episode_reward,7.464974e+03,2026-05-06T22:31:55
61,rl,recall,2,2,cumulative_profit,4.332225e+07,2026-05-06T22:31:55


Logged metrics:
model_type   cl_method  metric_name       
forecasting  ewc        mase                  3
                        rmse                  3
                        smape                 3
             naive      mase                  3
                        rmse                  3
                        smape                 3
             replay     mase                  3
                        rmse                  3
                        smape                 3
             sdft       mase                  3
                        rmse                  3
                        smape                 3
rl           ewc        avg_episode_reward    3
                        cumulative_profit     3
                        pricing_regret        3
             naive      avg_episode_reward    3
                        cumulative_profit     3
                        pricing_regret        3
             recall     avg_episode_reward    3
                        cumul

## Optional Summary Tables


In [9]:
cl_summary = build_cl_summary()
tables = print_and_save_comparison_tables(cl_summary)
cl_summary


CL Summary (BWT / FWT):

model_type cl_method    primary_metric  avg_final_perf  avg_online_perf  bwt  fwt
forecasting     naive              mase    2.049100e+00     2.682600e+00  0.0  0.0
forecasting       ewc              mase    3.256200e+00     3.269200e+00  0.0  0.0
forecasting    replay              mase    1.842500e+00     2.213400e+00  0.0  0.0
forecasting      sdft              mase    1.024600e+00     1.025200e+00  0.0  0.0
         rl     naive cumulative_profit    4.424801e+07     4.415691e+07  0.0  0.0
         rl       ewc cumulative_profit    4.374865e+07     4.384311e+07  0.0  0.0
         rl    recall cumulative_profit    4.278181e+07     4.326186e+07  0.0  0.0
         rl      sdft cumulative_profit    4.426396e+07     4.417387e+07  0.0  0.0


  FORECASTING - MASE (lower is better)
           Task 1  Task 2
cl_method                
ewc        2.9062  3.6191
naive      2.4678  2.2640
replay     2.1003  1.9556
sdft       1.5595  0.4903

  FORECASTING - sMAPE
             Task 1    Task 2
cl_method                    
ewc        104.4933  131.6897
naive       95.4880  111.2807
replay      91.0691  104.0968
sdft       100.1414  122.6640

  RL - CUMULATIVE PROFIT (higher is better)
                 Task 1        Task 2
cl_method                            
ewc        4.310701e+07  4.448476e+07
naive      4.329258e+07  4.511234e+07
recall     4.272142e+07  4.332225e+07
sdft       4.329156e+07  4.514628e+07

  RL - PRICING REGRET (lower is better)
            Task 1   Task 2
cl_method                  
ewc        31.8217  21.8476
naive      31.8056  21.8526
recall     31.8536  21.9645
sdft       31.8059  21.8422

  CL METRICS - BWT / FWT SUMMARY
 model_type cl_method    primary_metric  avg_final_perf  avg_online_perf  bwt  fwt
fo

,model_type,cl_method,primary_metric,avg_final_perf,avg_online_perf,bwt,fwt
0,forecasting,naive,mase,2.049100e+00,2.682600e+00,0.0,0.0
1,forecasting,ewc,mase,3.256200e+00,3.269200e+00,0.0,0.0
2,forecasting,replay,mase,1.842500e+00,2.213400e+00,0.0,0.0
3,forecasting,sdft,mase,1.024600e+00,1.025200e+00,0.0,0.0
4,rl,naive,cumulative_profit,4.424801e+07,4.415691e+07,0.0,0.0
5,rl,ewc,cumulative_profit,4.374865e+07,4.384311e+07,0.0,0.0
6,rl,recall,cumulative_profit,4.278181e+07,4.326186e+07,0.0,0.0
7,rl,sdft,cumulative_profit,4.426396e+07,4.417387e+07,0.0,0.0
